# 03 — Results Analysis

Post-training analysis of the best checkpoint:
- Test set metrics (accuracy, top-5, macro F1)
- Confusion matrix
- Per-class F1 bar chart
- Grad-CAM gallery
- Error analysis (most confident wrong predictions)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch

!pip install -q timm albumentations grad-cam

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader

DRIVE_ROOT  = Path('/content/drive/MyDrive/AI_TRAINING/skinDisease')
IMAGE_SIZE  = 224
NUM_CLASSES = 23
BACKBONE    = 'efficientnet_b0'
device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

proc      = DRIVE_ROOT / 'data' / 'processed'
ckpt_path = DRIVE_ROOT / 'outputs' / 'checkpoints' / 'best.pt'
fig_dir   = DRIVE_ROOT / 'outputs' / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)

# ---------- helpers (no src/ needed) ----------
def build_model(backbone, num_classes, dropout=0.4):
    model = timm.create_model(backbone, pretrained=False, num_classes=0)  # no head
    in_features = model.num_features
    model.classifier = torch.nn.Sequential(
        torch.nn.Dropout(dropout),
        torch.nn.Linear(in_features, num_classes),
    )
    return model

def get_val_transforms(size):
    return A.Compose([
        A.Resize(size, size),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

class SkinDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = cv2.cvtColor(cv2.imread(row['path']), cv2.COLOR_BGR2RGB)
        img = self.transform(image=img)['image']
        return img, int(row['label'])

# ---------- load checkpoint ----------
if not ckpt_path.exists():
    print('No checkpoint found — run training first.')
else:
    class_names = pd.read_csv(proc / 'classes.csv', index_col=0)['class_name'].tolist()
    ckpt  = torch.load(ckpt_path, map_location=device, weights_only=True)
    model = build_model(BACKBONE, NUM_CLASSES)
    model.load_state_dict(ckpt['model_state'])
    model.to(device).eval()
    print(f'Loaded checkpoint from epoch {ckpt["epoch"]}')

## 1. Test Set Metrics

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

test_df  = pd.read_csv(proc / 'test.csv')
val_df   = pd.read_csv(proc / 'val.csv')
train_df = pd.read_csv(proc / 'train.csv')

test_ds     = SkinDataset(test_df, get_val_transforms(IMAGE_SIZE))
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

all_preds, all_labels, all_probs = [], [], []
model.eval()
with torch.no_grad():
    for imgs, labels in test_loader:
        logits = model(imgs.to(device))
        probs  = torch.softmax(logits, dim=1).cpu()
        all_probs.append(probs)
        all_preds.extend(probs.argmax(1).tolist())
        all_labels.extend(labels.tolist())

all_probs = torch.cat(all_probs)

# Top-5 accuracy
top5 = sum(
    int(true) in torch.topk(prob, 5).indices.tolist()
    for prob, true in zip(all_probs, all_labels)
) / len(all_labels)

acc       = accuracy_score(all_labels, all_preds)
macro_f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
weighted_f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
report    = classification_report(all_labels, all_preds, target_names=class_names, zero_division=0)

print('=' * 55)
print('TEST SET RESULTS')
print(f'  Accuracy    : {acc:.4f}  ({acc*100:.1f}%)')
print(f'  Top-5 Acc   : {top5:.4f}  ({top5*100:.1f}%)')
print(f'  Macro F1    : {macro_f1:.4f}')
print(f'  Weighted F1 : {weighted_f1:.4f}')
print('=' * 55)

## 2. Full Classification Report

In [ ]:
print(report)

## 3. Confusion Matrix

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_labels, all_preds)
short_names = [n[:20] for n in class_names]

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues',
            xticklabels=short_names, yticklabels=short_names, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — Test Set')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.savefig(fig_dir / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Per-Class F1

In [ ]:
per_class_f1 = f1_score(all_labels, all_preds, average=None, zero_division=0)
f1_series = pd.Series(per_class_f1, index=class_names).sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#d73027' if v < 0.5 else '#4575b4' for v in f1_series]
f1_series.plot(kind='barh', ax=ax, color=colors)
ax.axvline(f1_series.mean(), color='black', linestyle='--',
           label=f'Mean F1 ({f1_series.mean():.2f})')
ax.set_xlabel('F1 Score'); ax.set_title('Per-Class F1 Score')
ax.legend(); plt.tight_layout()
plt.savefig(fig_dir / 'per_class_f1.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Grad-CAM Gallery

Visualises WHERE the model looks when making predictions.
Red/warm regions = highest influence on the prediction.

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import random

# Target the last conv block of EfficientNet
target_layer = [model.conv_head]

cam = GradCAM(model=model, target_layers=target_layer)

sample_indices = random.sample(range(len(test_ds)), 8)
MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(2, 8, figsize=(24, 6))
for col, idx in enumerate(sample_indices):
    tensor, label = test_ds[idx]
    input_t = tensor.unsqueeze(0).to(device)

    grayscale_cam = cam(input_tensor=input_t)[0]
    rgb_img = np.clip(tensor.permute(1,2,0).numpy() * STD + MEAN, 0, 1).astype(np.float32)
    cam_img = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

    axes[0, col].imshow(rgb_img)
    axes[0, col].axis('off')
    axes[0, col].set_title(class_names[label][:18], fontsize=6)

    axes[1, col].imshow(cam_img)
    axes[1, col].axis('off')
    axes[1, col].set_title('Grad-CAM', fontsize=6)

plt.suptitle('Grad-CAM — Model Attention (top=original, bottom=heatmap)', fontsize=12)
plt.tight_layout()
plt.savefig(fig_dir / 'gradcam_grid.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Error Analysis — Most Confident Wrong Predictions

In [ ]:
records = []
model.eval()
with torch.no_grad():
    for i in range(len(test_ds)):
        tensor, true_label = test_ds[i]
        logits = model(tensor.unsqueeze(0).to(device))
        probs  = torch.softmax(logits, dim=1).squeeze()
        pred_label = probs.argmax().item()
        confidence = probs[pred_label].item()
        if pred_label != true_label:
            records.append({
                'idx': i,
                'true': class_names[true_label],
                'pred': class_names[pred_label],
                'confidence': confidence,
                'path': test_df.iloc[i]['path'],
            })

errors_df = pd.DataFrame(records).sort_values('confidence', ascending=False)
print(f'Total errors : {len(errors_df)} / {len(test_ds)}')
print(f'Error rate   : {len(errors_df)/len(test_ds)*100:.1f}%')
print('\nTop-10 most confident wrong predictions:')
print(errors_df[['true', 'pred', 'confidence']].head(10).to_string(index=False))

In [ ]:
top_errors = errors_df.head(5)
fig, axes = plt.subplots(1, len(top_errors), figsize=(15, 4))
for ax, (_, row) in zip(axes, top_errors.iterrows()):
    img = cv2.cvtColor(cv2.imread(row['path']), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'True: {row["true"][:20]}\nPred: {row["pred"][:20]}\nConf: {row["confidence"]*100:.1f}%',
                 fontsize=7, color='red')
plt.suptitle('Most Confident Wrong Predictions', fontsize=12)
plt.tight_layout()
plt.savefig(fig_dir / 'top_errors.png', dpi=150)
plt.show()

## 7. Training History Curves

Loss and accuracy curves saved during training.

In [ ]:
from IPython.display import Image, display
for fig_name in ['loss_curves.png', 'accuracy_curves.png']:
    fig_path = fig_dir / fig_name
    if fig_path.exists():
        print(fig_name)
        display(Image(str(fig_path)))